In [5]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split


In [6]:

# Load dataset
df = pd.read_csv('../data/fer2013.csv')

# Map emotion labels to categories
emotion_labels = {
    0: 'Angry', 1: 'Disgust', 2: 'Fear', 3: 'Happy',
    4: 'Sad', 5: 'Surprise', 6: 'Neutral'
}

# Convert pixels to numpy arrays
pixels = df['pixels'].apply(lambda x: np.fromstring(x, sep=' '))
X = np.stack(pixels.values)
X = X.reshape(-1, 48, 48, 1).astype('float32') / 255.0

# One-hot encode labels
y = to_categorical(df['emotion'], num_classes=7)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (32298, 48, 48, 1)
Test shape: (3589, 48, 48, 1)


In [7]:
# Convert pixels to numpy arrays (fix deprecated np.fromstring)
pixels = df['pixels'].apply(lambda x: np.array(x.split(), dtype='float32'))
X = np.stack(pixels.values)
X = X.reshape(-1, 48, 48, 1).astype('float32') / 255.0

In [8]:

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(48,48,1)),
    MaxPooling2D(pool_size=(2,2)),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(7, activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()





Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 46, 46, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 23, 23, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 21, 21, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 10, 10, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 8, 8, 128)         73856     
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 4, 4, 128)       

In [9]:

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=64
)


Epoch 1/20


505/505 [==============================] - 28s 51ms/step - loss: 1.6677 - accuracy: 0.3366 - val_loss: 1.4576 - val_accuracy: 0.4394
Epoch 2/20
505/505 [==============================] - 32s 64ms/step - loss: 1.4040 - accuracy: 0.4641 - val_loss: 1.3178 - val_accuracy: 0.4979
Epoch 3/20
505/505 [==============================] - 32s 64ms/step - loss: 1.2842 - accuracy: 0.5124 - val_loss: 1.2238 - val_accuracy: 0.5417
Epoch 4/20
505/505 [==============================] - 18s 36ms/step - loss: 1.2114 - accuracy: 0.5440 - val_loss: 1.1783 - val_accuracy: 0.5534
Epoch 5/20
505/505 [==============================] - 19s 38ms/step - loss: 1.1450 - accuracy: 0.5677 - val_loss: 1.1505 - val_accuracy: 0.5704
Epoch 6/20
505/505 [==============================] - 17s 33ms/step - loss: 1.0907 - accuracy: 0.5875 - val_loss: 1.1307 - val_accuracy: 0.5798
Epoch 7/20
505/505 [==============================] - 26s 52ms/step - loss: 1.0394 - accuracy: 0.6123 - val_loss: 1.1245 - val_accurac

In [10]:

model.save('../models/emotion_cnn.h5')
print("Model saved to ../models/emotion_cnn.h5")


Model saved to ../models/emotion_cnn.h5


c:\Users\singh\OneDrive\Desktop\EmotionSense\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
